# 🧠 RAG Masterclass: From Zero to Production
## A Complete 5-Stage Learning Journey in Retrieval-Augmented Generation

---

> **About this notebook**
>
> This notebook takes you through the full arc of building a RAG system — from first principles
> to production-grade techniques. Each stage has a clear goal, theory section, working code,
> and a mini-exercise to cement your understanding.
>
> **What you will build:** A question-answering assistant for *MediVault*, a fictional
> medical-records SaaS company. The assistant must accurately answer HR, legal, product,
> and policy questions using only company documents.
>
> **Why RAG?** Language models hallucinate when asked about facts they haven't seen.
> RAG plugs in a retrieval layer so the model reads the right documents before answering —
> giving you accuracy without the cost of fine-tuning.

---

| Stage | Title | Core Skill |
|-------|-------|-----------|
| 1 | Foundations — How RAG Actually Works | Semantic search from scratch |
| 2 | Vector Databases & Embeddings at Scale | ChromaDB + HuggingFace embeddings |
| 3 | Building a Conversational RAG Pipeline | LangChain LCEL chains |
| 4 | Evaluating RAG — Metrics That Matter | RAGAS-style automated evaluation |
| 5 | Advanced RAG — Reranking & Query Rewriting | Production hardening |

---

### Prerequisites
```bash
pip install openai chromadb langchain langchain-openai langchain-chroma \
            langchain-huggingface sentence-transformers tiktoken \
            plotly scikit-learn tqdm pydantic python-dotenv gradio
```
** These are already in dependencies. You do NOT need to run the above pip install on local. Run it when you are running it on Google Colab.


---
# 🟢 Stage 1 — Foundations: How RAG Actually Works

## 🎯 Target
Understand the *mechanics* of RAG without any framework magic.
By the end of this stage you will:
- Know what an embedding is and why semantic search beats keyword search
- Implement cosine-similarity retrieval from scratch in pure Python/NumPy
- Build a minimal but *working* RAG pipeline in under 50 lines of code

---

## 📖 Theory: The Problem With Prompt-Stuffing

The naive approach to giving an LLM company knowledge is to dump everything into the prompt:

```
system: You are a MediVault assistant. Here is every policy doc we have: [all 300 pages] ...
user: What is the parental-leave policy?
```

**Why this fails:**
1. **Context limits** — GPT-4o's 128k window sounds big until you have 500 policy docs.
2. **Cost** — You pay per token. Sending 50k tokens every call is expensive.
3. **Attention dilution** — LLMs perform worse when the relevant fact is buried in noise
   (the "lost in the middle" problem, Liu et al., 2023).

**The RAG fix:** Retrieve *only* the 3-5 most relevant document chunks and inject those.

---

## 📖 Theory: Embeddings & Semantic Search

An **embedding** is a fixed-length numerical vector that encodes *meaning*.
Two sentences with similar meaning will have vectors that point in nearly the same direction.

```
embed("What is the refund policy?")  →  [0.12, -0.87, 0.44, ...]   (1536 numbers)
embed("How do I get my money back?") →  [0.13, -0.85, 0.41, ...]   ← similar direction!
embed("What is the CEO's name?")     →  [-0.34, 0.22, -0.71, ...]  ← different direction
```

**Cosine similarity** measures the angle between two vectors:
```
similarity = (A · B) / (|A| × |B|)   →   range: -1 (opposite) to 1 (identical)
```

Keyword search would miss `"refund"` ↔ `"money back"`. Embedding search catches it.

---


In [ ]:
# Stage 1 setup — no frameworks, pure Python + OpenAI

import os
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"   # cheap, fast, 1536 dims
CHAT_MODEL  = "gpt-4o-mini"

print("✅ OpenAI client ready")


### 1.1  Our Mini Knowledge Base

We'll use six short documents from *MediVault*'s internal wiki.
Each is deliberately brief so you can read them all and verify the retrieval is correct.


In [ ]:
# Six MediVault documents (in a real project these come from files)

DOCUMENTS = [
    {
        "id": "hr-001",
        "title": "Parental Leave Policy",
        "text": (
            "MediVault offers 16 weeks of fully paid parental leave for primary caregivers "
            "and 6 weeks for secondary caregivers. Leave may begin up to 4 weeks before the "
            "expected birth date. Adoption and surrogacy qualify on the same terms. "
            "Employees must give 30 days notice where possible. Leave is job-protected."
        )
    },
    {
        "id": "hr-002",
        "title": "Remote Work Policy",
        "text": (
            "Employees may work remotely up to 3 days per week with manager approval. "
            "Core collaboration hours are 10am-3pm in the employee's local timezone. "
            "A fully remote arrangement requires VP-level sign-off and is reviewed annually. "
            "The company provides a $500 annual home-office stipend."
        )
    },
    {
        "id": "prod-001",
        "title": "MediVault Core — Product Overview",
        "text": (
            "MediVault Core is a HIPAA-compliant electronic health records (EHR) platform. "
            "It supports HL7 FHIR R4, DICOM imaging, and real-time bidirectional EHR sync. "
            "Pricing starts at $18 per provider per month. SOC-2 Type II certified. "
            "Mobile apps are available on iOS and Android."
        )
    },
    {
        "id": "prod-002",
        "title": "MediVault Analytics — Product Overview",
        "text": (
            "MediVault Analytics adds population health dashboards, predictive readmission "
            "risk scoring, and custom report builder to the Core platform. "
            "Requires MediVault Core. Additional $12 per provider per month. "
            "Data is retained for 7 years and exportable in CSV, FHIR JSON, and Parquet."
        )
    },
    {
        "id": "legal-001",
        "title": "Data Processing Agreement Summary",
        "text": (
            "MediVault acts as a Business Associate under HIPAA for all covered entities. "
            "PHI is encrypted at rest (AES-256) and in transit (TLS 1.3). "
            "Sub-processors are listed in Annex B and updated 30 days in advance of changes. "
            "Breach notification is provided within 60 hours of discovery, meeting HIPAA requirements."
        )
    },
    {
        "id": "legal-002",
        "title": "Acceptable Use Policy",
        "text": (
            "MediVault systems may not be used for unauthorized access, data exfiltration, "
            "or any activity violating HIPAA or applicable law. "
            "Security incidents must be reported to security@medivault.io within 24 hours. "
            "Violations may result in immediate termination and legal action."
        )
    },
]

print(f"Knowledge base loaded: {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  [{doc['id']}] {doc['title']} — {len(doc['text'])} chars")


### 1.2  Embedding the Documents

We call the OpenAI Embeddings API once per document and cache the vectors in a Python dict.


In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    """Embed a batch of texts; return list of float vectors."""
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


# Embed all documents in one API call (batching = fewer round-trips)
doc_texts = [doc["text"] for doc in DOCUMENTS]
doc_vectors = embed(doc_texts)

# Store alongside documents for easy lookup
for doc, vec in zip(DOCUMENTS, doc_vectors):
    doc["vector"] = np.array(vec)

print(f"✅ Embedded {len(DOCUMENTS)} documents")
print(f"   Vector dimension: {len(DOCUMENTS[0]['vector'])}")
# The "vector" key is now added to each document dictionary in DOCUMENTS, allowing easy access to both text and vector for each document when needed.
#print(DOCUMENTS[0]["vector"][:10])  # first 10 dimensions
#print(DOCUMENTS[0])


### 1.3  Cosine Similarity From Scratch

Let's implement retrieval manually so the math is transparent.


In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors (range -1 to 1)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """
    Embed the query, then rank all documents by cosine similarity.
    Returns the top_k most relevant documents with their scores.
    """
    query_vec = np.array(embed([query])[0])

    scored = []
    for doc in DOCUMENTS:
        score = cosine_similarity(query_vec, doc["vector"])
        scored.append({"score": score, "doc": doc})

    #print(scored)
    scored.sort(key=lambda x: x["score"], reverse=True)
    #print(scored)
    return scored[:top_k]


# ── Test retrieval ───────────────────────────────────────────────────────────
query = "How long is parental leave?"
results = retrieve(query)

print(f"Query: '{query}'\n")
print(f"{'Score':>6}  Document")
print("─" * 60)
for r in results:
    print(f"{r['score']:6.4f}  [{r['doc']['id']}] {r['doc']['title']}")


### 1.4  The RAG Pipeline — From Scratch

Now connect retrieval → prompt construction → LLM answer.


In [ ]:
SYSTEM_PROMPT = """You are a helpful internal assistant for MediVault, a healthcare SaaS company.
Answer the user's question using ONLY the context provided below.
If the context does not contain the answer, say "I don't have that information."
Be concise and cite the document title you used.

Context:
{context}
"""

def rag_answer(question: str, top_k: int = 3) -> dict:
    """
    Full RAG pipeline:
      1. Retrieve relevant chunks
      2. Build a context-injected prompt
      3. Generate an answer with the LLM
    Returns a dict with the answer and retrieved docs for inspection.
    """
    # Step 1 — Retrieve
    results = retrieve(question, top_k=top_k)
    context_parts = []
    for r in results:
        doc = r["doc"]
        context_parts.append(f"[{doc['title']}]\n{doc['text']}")
    context = "\n\n".join(context_parts)

    print(f"Context for question: '{question}'\n")
    print(context)
    print()

    # Step 2 — Prompt
    system = SYSTEM_PROMPT.format(context=context)

    # Step 3 — Generate
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": question},
        ],
        temperature=0,
    )
    answer = response.choices[0].message.content

    return {
        "question": question,
        "answer":   answer,
        "sources":  [(r["doc"]["id"], r["doc"]["title"], round(r["score"], 4)) for r in results],
    }


# ── Run a few test questions ─────────────────────────────────────────────────
test_questions = [
    "How many weeks of parental leave does MediVault offer primary caregivers?",
    "What encryption standard does MediVault use for data at rest?",
    "Can I work from home every day?",
    "What is the monthly cost of MediVault Analytics?",
]

for q in test_questions:
    result = rag_answer(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"   Sources: {result['sources']}")
    print()


### 🔍 Stage 1 Insight: What Just Happened?

```
User question
     │
     ▼  embed()
Query vector  ──cosine_sim──►  Ranked document list
                                        │
                                        ▼  top 3 docs
                                 Context string
                                        │
                                        ▼  LLM
                                    Answer
```

**Key observations:**
- The retrieval took ~200ms; the LLM call took ~1s. Retrieval is the fast part.
- We sent only ~300 tokens of context instead of the full 6-document corpus.
- The LLM correctly refused when the context didn't have the answer.

---

### 🏋️ Stage 1 Exercise

Modify `retrieve()` to accept a `min_score` threshold — if no document exceeds the threshold,
return an empty list. Then update `rag_answer()` to handle the empty case gracefully
(print "I have no relevant documents" rather than hallucinating from an empty context).

<details>
<summary>Hint</summary>

```python
def retrieve(query, top_k=3, min_score=0.3):
    ...
    return [r for r in scored[:top_k] if r["score"] >= min_score]
```

</details>


---
# 🔵 Stage 2 — Vector Databases & Embeddings at Scale

## 🎯 Target
Move from an in-memory dict to a *persistent vector database* capable of handling
thousands of documents. By the end of this stage you will:
- Understand chunking strategies and why they matter
- Persist embeddings in ChromaDB and query them without recomputing
- Visualise your vector space to build intuition about what embeddings capture

---

## 📖 Theory: Why We Chunk

Real documents are long. Embedding a 10-page policy doc produces one vector that
averages everything — retrieval gets blurry. **Chunking** splits documents into
smaller passages so each vector encodes a *specific* topic.

**The chunking trade-off:**

| Chunk size | Pro | Con |
|-----------|-----|-----|
| Large (1000+ tokens) | Full context, fewer chunks | Noisy embedding, retrieval imprecision |
| Small (100 tokens) | Precise embedding | May lack context for answering |
| **Medium (300-500 tokens) + overlap** | **Balance** | **Sweet spot for most RAG** |

**Overlap** (e.g. 10-20%) repeats the last N words of chunk *i* at the start of chunk *i+1*,
so a fact straddling a boundary still appears in at least one chunk.

---

## 📖 Theory: Choosing an Embedding Model

| Model | Dim | Speed | Cost | Best for |
|-------|-----|-------|------|---------|
| `text-embedding-3-small` | 1536 | Fast | $0.02/1M tokens | General RAG |
| `text-embedding-3-large` | 3072 | Med | $0.13/1M tokens | High-stakes retrieval |
| `all-MiniLM-L6-v2` (HF) | 384 | Very fast | Free | Local/offline |
| `BAAI/bge-large-en-v1.5` (HF) | 1024 | Med | Free | Best free option |

A smaller dimension is not always worse — `3-small` often matches `3-large` for RAG
because retrieval is a ranking problem, not a precision-of-individual-coordinates problem.

---


In [ ]:
# Stage 2 setup

import os
import glob
import textwrap
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import plotly.graph_objects as go
from sklearn.manifold import TSNE

load_dotenv(override=True)
print("✅ Libraries ready")


### 2.1  A Realistic Document Set

We simulate loading markdown files from a directory tree. In your real project
you'd point this at an actual folder.


In [ ]:
# Simulated documents — longer, more realistic than Stage 1
# (In production: glob("knowledge-base/**/*.md", recursive=True))

RAW_DOCS = {
    "hr/parental-leave.md": """
# Parental Leave Policy — MediVault Inc.

## Eligibility
All full-time employees who have completed 90 days of service are eligible.
Part-time employees (>20h/week) are eligible after 6 months.

## Leave Entitlements
- **Primary caregiver**: 16 weeks fully paid, extendable by 4 weeks unpaid.
- **Secondary caregiver**: 6 weeks fully paid.
- **Adoption / surrogacy**: Same entitlements as biological birth.

## How to Apply
Submit a Parental Leave Request form in Workday at least 30 days before
the anticipated start date. Attach the relevant documentation (birth certificate
preview, adoption letter, or surrogacy agreement).

## Pay During Leave
Base salary continues. Variable compensation (bonuses, commissions) is prorated
based on the employee's trailing 3-month average.

## Benefits Continuation
Health, dental, and vision benefits continue unchanged throughout the leave period.
401k contributions pause but the vesting clock does not.

## Return to Work
Employees return to the same or equivalent role. A 2-week phased re-entry
(50% schedule) is available on request.
""",

    "hr/remote-work.md": """
# Remote Work Policy — MediVault Inc.

## Hybrid Default
The default arrangement for all non-lab roles is hybrid: minimum 2 in-office days
per week (Tuesday and Thursday are designated anchor days).

## Full Remote
Approved on a case-by-case basis by the VP of People. Criteria include:
- Role does not require physical presence or in-person collaboration.
- Employee has been with MediVault for at least 12 months.
- Manager endorsement.
Full-remote employees must visit HQ at least once per quarter.

## Core Hours
All employees, regardless of location, must be reachable between 10am–3pm in their
local timezone on business days.

## Equipment and Stipends
- Company laptop provided (MacBook Pro 14" or Windows equivalent).
- $500 annual home-office stipend, receipts required.
- Internet subsidy: $50/month, applied as payroll addition.

## Security Requirements
Remote employees must use the company VPN for all work. Personal devices may not
be used to process PHI.
""",

    "products/medivault-core.md": """
# MediVault Core — Product Specification

## Overview
MediVault Core is a cloud-native, HIPAA-compliant Electronic Health Records (EHR)
platform designed for outpatient clinics and hospital networks of all sizes.

## Key Features
- **HL7 FHIR R4** API for interoperability with any modern health system.
- **DICOM Viewer** — embedded imaging with 3D reconstruction.
- **Scheduling & Billing** — integrated appointment management and ICD-10 coding.
- **Patient Portal** — secure messaging, results sharing, and e-prescribing.
- **Mobile** — iOS and Android apps; offline mode with sync on reconnect.

## Compliance
SOC-2 Type II | HIPAA Business Associate | ONC Certified EHR Technology (CEHRT)
ISO 27001 in progress (expected Q3).

## Pricing
$18 per active provider per month (billed annually).
$22 per provider per month (billed monthly).
Volume discounts available for networks above 50 providers.

## Infrastructure
Hosted on AWS (us-east-1, us-west-2, eu-west-1). 99.9% SLA. Automated daily backups
with 35-day retention. Disaster recovery RTO < 4 hours.
""",

    "products/medivault-analytics.md": """
# MediVault Analytics — Product Specification

## Overview
MediVault Analytics is an add-on module for MediVault Core that delivers population
health intelligence and operational dashboards.

## Requirements
Active MediVault Core subscription. Analytics data lags Core by up to 15 minutes
(near-real-time streaming pipeline via Apache Kafka).

## Features
- **Population Health Dashboard**: Chronic disease cohorts, gap-in-care alerts.
- **Readmission Risk Score**: ML model (XGBoost) trained on 10M+ patient encounters;
  AUC 0.81 on held-out validation set.
- **Custom Report Builder**: Drag-and-drop, exportable as PDF, CSV, FHIR JSON, Parquet.
- **Benchmarking**: Compare your clinic's metrics against anonymised regional peers.

## Pricing
$12 per active provider per month (requires Core subscription).

## Data Retention & Export
7-year retention (exceeds HIPAA's 6-year minimum).
Full data export available on contract termination within 30 days.
""",

    "legal/dpa.md": """
# Data Processing Agreement — Summary Sheet

## Role Classification
MediVault acts as a **Business Associate** (BA) under HIPAA for all covered entities.
Customers acting as a Covered Entity must execute a Business Associate Agreement (BAA)
before going live. MediVault provides a standard BAA template; custom BAAs are
reviewed by legal within 10 business days.

## Security Controls
- Encryption at rest: AES-256.
- Encryption in transit: TLS 1.3 minimum.
- Access control: Role-based, least-privilege; MFA enforced for all admin accounts.
- Penetration testing: Annual third-party; results summary available under NDA.

## Sub-processors
Listed in Annex B of the DPA. Customers are notified 30 days before any new
sub-processor is added. Opt-out window is 15 days.

## Breach Notification
MediVault will notify the customer within 60 hours of discovering a breach
affecting their PHI. This exceeds HIPAA's 60-day requirement significantly.

## Audits
Customers may request audit logs for their data at any time via the Admin Console.
Third-party audit reports (SOC-2) are available on request.
""",
}

print(f"Loaded {len(RAW_DOCS)} documents")
for path, content in RAW_DOCS.items():
    print(f"  {path}: {len(content)} chars")


### 2.2  Recursive Character Chunking

We implement chunking from scratch — it's just a few lines — so you understand
exactly what LangChain's `RecursiveCharacterTextSplitter` is doing.


In [ ]:
def chunk_text(
    text: str,
    chunk_size: int = 400,
    chunk_overlap: int = 80,
    separators: list[str] = ["\n\n", "\n", ". ", " "],
) -> list[str]:
    """
    Recursively split text on separators, then enforce chunk_size.
    Overlap is added by re-including the last `chunk_overlap` characters
    of the previous chunk at the start of the next.

    This mirrors LangChain's RecursiveCharacterTextSplitter logic.
    """
    def split_on_separator(text, sep):
        return text.split(sep) if sep else list(text)

    # Try each separator in order until chunks are small enough
    for sep in separators:
        parts = split_on_separator(text, sep)
        if max(len(p) for p in parts) <= chunk_size:
            break

    # Merge parts back up to chunk_size with overlap
    chunks, current, current_len = [], "", 0
    for part in parts:
        part_with_sep = (sep + part) if current else part
        if current_len + len(part_with_sep) > chunk_size and current:
            chunks.append(current.strip())
            # Overlap: keep last `chunk_overlap` chars of current
            overlap_start = max(0, len(current) - chunk_overlap)
            current = current[overlap_start:] + part_with_sep
            current_len = len(current)
        else:
            current += part_with_sep
            current_len += len(part_with_sep)
    if current.strip():
        chunks.append(current.strip())

    return chunks


# ── Chunk all documents ──────────────────────────────────────────────────────
all_chunks = []

for source_path, text in RAW_DOCS.items():
    doc_type = source_path.split("/")[0]        # "hr", "products", "legal"
    doc_chunks = chunk_text(text, chunk_size=400, chunk_overlap=80)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "text":     chunk,
            "source":   source_path,
            "doc_type": doc_type,
            "chunk_id": f"{source_path}::{i}",
        })

print(f"Total chunks created: {len(all_chunks)}")
print()
# Show chunk size distribution
sizes = [len(c["text"]) for c in all_chunks]
print(f"Chunk sizes — min: {min(sizes)}, max: {max(sizes)}, avg: {sum(sizes)//len(sizes)}")
print()
print("Sample chunk:")
print("─" * 60)
print(all_chunks[2]["text"])


### 2.3  Storing in ChromaDB

ChromaDB is a lightweight, embeddable vector database that runs in-process.
No server to start, no Docker required.


In [ ]:
# We use the free, local SentenceTransformer embedding model
# so this stage runs without any API key

from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"   # 384-dim, very fast, excellent for RAG demos
model = SentenceTransformer(EMBED_MODEL_NAME)

print(f"✅ Embedding model loaded: {EMBED_MODEL_NAME}")
print(f"   Output dimension: {model.get_sentence_embedding_dimension()}")


In [ ]:
# Create (or reset) a persistent ChromaDB collection

DB_PATH = "/tmp/medivault_chroma"
COLLECTION_NAME = "knowledge_base"

chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Delete old collection if it exists (for clean re-runs)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("♻️  Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},    # use cosine distance
)

print(f"✅ Collection '{COLLECTION_NAME}' created")


In [ ]:
# Embed all chunks and insert into ChromaDB

texts     = [c["text"]     for c in all_chunks]
ids       = [c["chunk_id"] for c in all_chunks]
metadatas = [{"source": c["source"], "doc_type": c["doc_type"]} for c in all_chunks]

# Batch-embed with SentenceTransformer (fast, local)
vectors = model.encode(texts, show_progress_bar=True, batch_size=32).tolist()

collection.add(
    ids=ids,
    embeddings=vectors,
    documents=texts,
    metadatas=metadatas,
)

print(f"\n✅ Inserted {collection.count()} chunks into ChromaDB")

# pretty print the contents of collection
data = collection.get()

for i in range(len(data["ids"])):
    print(f"ID: {data['ids'][i]}")
    print(f"Document: {data['documents'][i]}")
    print(f"Metadata: {data['metadatas'][i]}")
    print("-" * 80)


### 2.4  Querying ChromaDB

Now retrieval is a single `.query()` call — the vector DB handles ANN (Approximate Nearest Neighbour) search.


In [ ]:
def chroma_retrieve(query: str, top_k: int = 4) -> list[dict]:
    """Embed query and search ChromaDB; return top_k results."""
    query_vec = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    output = []
    for i in range(len(results["ids"][0])):
        output.append({
            "text":     results["documents"][0][i],
            "source":   results["metadatas"][0][i]["source"],
            "doc_type": results["metadatas"][0][i]["doc_type"],
            "distance": results["distances"][0][i],       # lower = more similar (cosine)
            "similarity": 1 - results["distances"][0][i], # convert to similarity
        })
    return output


# ── Test queries ─────────────────────────────────────────────────────────────
test_queries = [
    "What is the encryption standard for data at rest?",
    "Can engineers work fully remote?",
    "What machine learning model powers readmission prediction?",
]

for query in test_queries:
    print(f"🔍 Query: {query}")
    hits = chroma_retrieve(query, top_k=2)
    for h in hits:
        print(f"   [{h['similarity']:.3f}] {h['source']}")
        print(f"   {h['text'][:120]}...")
    print()


### 2.5  Visualising the Embedding Space

t-SNE reduces our 384-dimensional vectors to 2D so we can see how ChromaDB
has organised the knowledge.


In [ ]:
# Pull all vectors from ChromaDB for visualisation
raw = collection.get(include=["embeddings", "documents", "metadatas"])
vecs   = np.array(raw["embeddings"])
docs   = raw["documents"]
metas  = raw["metadatas"]

# Reduce to 2D with t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(10, len(vecs)-1))
coords = tsne.fit_transform(vecs)

# Colour by document type
TYPE_COLORS = {"hr": "#3B82F6", "products": "#10B981", "legal": "#F59E0B"}
colors  = [TYPE_COLORS.get(m["doc_type"], "#6B7280") for m in metas]
labels  = [m["doc_type"] for m in metas]
sources = [m["source"] for m in metas]

fig = go.Figure()

for doc_type, color in TYPE_COLORS.items():
    mask = [i for i, l in enumerate(labels) if l == doc_type]
    fig.add_trace(go.Scatter(
        x=coords[mask, 0],
        y=coords[mask, 1],
        mode="markers",
        name=doc_type.upper(),
        marker=dict(color=color, size=10, opacity=0.85,
                    line=dict(width=1, color="white")),
        text=[f"<b>{sources[i]}</b><br>{docs[i][:80]}..." for i in mask],
        hoverinfo="text",
    ))

fig.update_layout(
    title="🗺️ MediVault Knowledge Base — Embedding Space (t-SNE 2D)",
    plot_bgcolor="#0f172a",
    paper_bgcolor="#0f172a",
    font=dict(color="white"),
    legend=dict(bgcolor="#1e293b"),
    xaxis=dict(showgrid=False, zeroline=False),
    yaxis=dict(showgrid=False, zeroline=False),
    width=800, height=550,
)
fig.show()
print("💡 Notice how chunks from the same document cluster together!")


### 🏋️ Stage 2 Exercise

Experiment with different chunk sizes. Create two collections:
- `chunk_size=150, overlap=30`
- `chunk_size=600, overlap=120`

For the query `"How long must MediVault notify customers before adding a sub-processor?"`:
1. Retrieve top-3 chunks from each collection.
2. Which chunk size returns the most precisely targeted passage?
3. Print the exact text that contains the answer.

**Bonus**: Try swapping to `BAAI/bge-base-en-v1.5` — does retrieval improve?


---
# 🟡 Stage 3 — Building a Conversational RAG Pipeline

## 🎯 Target
Build a *production-quality*, multi-turn conversational RAG pipeline.
By the end of this stage you will:
- Understand why naïve RAG breaks in multi-turn conversations
- Implement history-aware query rewriting using an LLM
- Build an end-to-end pipeline with LangChain LCEL (`|` chains)
- Launch a working chat UI with Gradio

---

## 📖 Theory: The Multi-Turn Problem

Single-turn RAG is easy. But conversations are stateful:

```
User: What's the parental leave for primary caregivers?
Bot:  16 weeks paid.
User: What about secondary?           ← "What about" refers to parental leave — but
                                         how does the retriever know that?
```

If you embed `"What about secondary?"` directly, the retriever has no idea what
"secondary" refers to. You'll get a bad retrieval match.

**Solution: Contextual Query Rewriting**

Before retrieval, pass the conversation history + new message to an LLM and ask it
to produce a *standalone* search query:

```
Conversation so far: [parental leave question + answer]
New message: "What about secondary?"
→ Rewritten query: "parental leave secondary caregiver MediVault"
```

This single step dramatically improves multi-turn RAG accuracy.

---

## 📖 Theory: LangChain LCEL

LangChain Expression Language (LCEL) lets you compose pipeline steps with `|` (pipe):

```python
chain = step_A | step_B | step_C
result = chain.invoke({"input": "..."})
```

Each step receives the output of the previous step. LCEL handles streaming, async,
batching, and tracing automatically.

---


In [ ]:
# Stage 3 setup — LangChain LCEL pipeline

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import gradio as gr

load_dotenv(override=True)

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL  = "all-MiniLM-L6-v2"   # free local embeddings
DB_PATH      = "/tmp/medivault_lc"

print("✅ LangChain components imported")


In [ ]:
# ── Build the vector store from our documents ────────────────────────────────
# (same documents as Stage 1/2, now via LangChain Document objects)

# All five MediVault documents defined inline so this cell is self-contained
RAW_DOCS_LC = {
    "hr/parental-leave.md": (
        "# Parental Leave Policy\n\n"
        "All full-time employees with 90 days tenure are eligible. "
        "Primary caregivers receive 16 weeks fully paid; secondary caregivers receive 6 weeks. "
        "Adoption and surrogacy qualify on identical terms. "
        "Requests submitted in Workday 30 days before leave start. "
        "Base salary continues; variable pay is prorated. "
        "Benefits continue throughout. Phased return (50% for 2 weeks) available on request."
    ),
    "hr/remote-work.md": (
        "# Remote Work Policy\n\n"
        "Hybrid default: 2 in-office days per week (Tue/Thu anchor days). "
        "Full-remote requires VP of People approval, 12 months tenure, and manager endorsement. "
        "Core hours: 10am-3pm local time. "
        "Stipend: $500 annual home-office, $50/month internet subsidy. "
        "Must use VPN and company devices for PHI."
    ),
    "products/medivault-core.md": (
        "# MediVault Core\n\n"
        "HIPAA-compliant EHR platform. HL7 FHIR R4, DICOM, scheduling, billing, "
        "patient portal, iOS/Android apps. SOC-2 Type II, ONC CEHRT. "
        "$18/provider/month annual; $22 monthly. Volume discounts above 50 providers. "
        "AWS hosted, 99.9% SLA, 35-day backup retention, RTO < 4 hours."
    ),
    "products/medivault-analytics.md": (
        "# MediVault Analytics\n\n"
        "Add-on to Core. Population health dashboards, XGBoost readmission risk (AUC 0.81), "
        "custom report builder (PDF/CSV/FHIR/Parquet), regional benchmarking. "
        "Near-real-time Kafka pipeline. $12/provider/month. "
        "7-year data retention, full export within 30 days of termination."
    ),
    "legal/dpa.md": (
        "# Data Processing Agreement Summary\n\n"
        "MediVault is a HIPAA Business Associate. BAA required before go-live; "
        "standard template available. AES-256 at rest, TLS 1.3 in transit. MFA enforced. "
        "Annual third-party pen testing. Sub-processor changes notified 30 days in advance; "
        "15-day opt-out window. Breach notification within 60 hours. SOC-2 reports available on request."
    ),
}

# Build LangChain Document objects
lc_docs = [
    Document(
        page_content=text,
        metadata={"source": path, "doc_type": path.split("/")[0]},
    )
    for path, text in RAW_DOCS_LC.items()
]

# Chunk with RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=70)
chunks = splitter.split_documents(lc_docs)
print(f"Created {len(chunks)} chunks from {len(lc_docs)} documents")


In [ ]:
# Build persistent Chroma vector store (HuggingFace local embeddings)

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# Delete old store if it exists
import shutil, pathlib
if pathlib.Path(DB_PATH).exists():
    shutil.rmtree(DB_PATH)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_PATH,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✅ Vector store ready: {vectorstore._collection.count()} vectors")


### 3.1  History-Aware Query Rewriting

This is the key ingredient for good multi-turn RAG.


In [ ]:
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# ── Step 1: Rewrite the user's message into a standalone search query ──────

REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are a query rewriter for a RAG system.
Given a conversation history and the latest user message, rewrite the user's message
into a single, self-contained search query that can be used to retrieve relevant documents.
The query should capture the user's actual information need even if their message uses
pronouns or references previous turns.
Return ONLY the rewritten query — no preamble, no explanation."""),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

rewrite_chain = REWRITE_PROMPT | llm | StrOutputParser()

# ── Test the rewriter ─────────────────────────────────────────────────────

history_example = [
    HumanMessage("What parental leave do primary caregivers get?"),
    AIMessage("Primary caregivers at MediVault receive 16 weeks of fully paid parental leave."),
]
followup = "And secondary caregivers?"

rewritten = rewrite_chain.invoke({"history": history_example, "question": followup})
print(f"Original : {followup!r}")
print(f"Rewritten: {rewritten!r}")
print()

history_example2 = [
    HumanMessage("Tell me about MediVault Core pricing."),
    AIMessage("MediVault Core costs $18 per provider per month billed annually."),
]
followup2 = "Does it include analytics?"
rewritten2 = rewrite_chain.invoke({"history": history_example2, "question": followup2})
print(f"Original : {followup2!r}")
print(f"Rewritten: {rewritten2!r}")


### 3.2  The Full LCEL Conversational Chain


In [ ]:
# ── Step 2: Format retrieved documents into a context string ─────────────

def format_docs(docs):
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        parts.append(f"[Source: {source}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


# ── Step 3: Answer generation prompt ──────────────────────────────────────

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are a knowledgeable internal assistant for MediVault, a healthcare SaaS company.
Answer the user's question using ONLY the retrieved context below.
Always cite the source document in your answer (e.g., "According to the Remote Work Policy...").
If the context does not contain sufficient information, say so honestly — do not guess.

Retrieved context:
{context}"""),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])


# ── Compose the full chain with LCEL ──────────────────────────────────────
#
#  Input dict: {"question": str, "history": list[BaseMessage]}
#
#  Pipe steps:
#    1. Rewrite question → standalone_query
#    2. Retrieve with standalone_query
#    3. Format docs → context string
#    4. Feed context + history + original question into answer LLM

def build_rag_chain():
    # We need to thread both the original question (for the answer prompt)
    # and the rewritten query (for retrieval) through the pipe.
    # RunnablePassthrough.assign() lets us add keys without losing others.

    retrieval_chain = (
        RunnablePassthrough.assign(
            standalone_query=rewrite_chain
        )
        | RunnablePassthrough.assign(
            context=RunnableLambda(lambda x: format_docs(
                retriever.invoke(x["standalone_query"])
            ))
        )
        | ANSWER_PROMPT
        | llm
        | StrOutputParser()
    )
    return retrieval_chain


rag_chain = build_rag_chain()
print("✅ RAG chain assembled")


In [ ]:
# ── Single-turn test ──────────────────────────────────────────────────────

response = rag_chain.invoke({
    "question": "What encryption does MediVault use for data at rest?",
    "history":  [],
})
print(response)


In [ ]:
# ── Multi-turn test ───────────────────────────────────────────────────────

conversation_history = []

def chat_turn(question: str) -> str:
    """Run one turn of conversation; update history in place."""
    answer = rag_chain.invoke({
        "question": question,
        "history":  conversation_history,
    })
    conversation_history.append(HumanMessage(content=question))
    conversation_history.append(AIMessage(content=answer))
    return answer


turns = [
    "What is included in MediVault Core?",
    "How much does it cost?",                      # pronoun — needs rewriting
    "And the analytics add-on?",                   # follow-up — needs context
    "Is the analytics data retention HIPAA compliant?",
]

for turn in turns:
    print(f"User: {turn}")
    answer = chat_turn(turn)
    print(f"Bot:  {answer}")
    print()


### 3.3  Gradio Chat UI


In [ ]:
def gradio_chat(message: str, history: list) -> str:
    """
    Gradio-compatible chat function.
    `history` is a list of [user_msg, bot_msg] pairs from Gradio.
    We convert to LangChain message format before calling the chain.
    """
    lc_history = []
    for user_msg, bot_msg in history:
        lc_history.append(HumanMessage(content=user_msg))
        if bot_msg:
            lc_history.append(AIMessage(content=bot_msg))

    answer = rag_chain.invoke({
        "question": message,
        "history":  lc_history,
    })
    return answer


demo = gr.ChatInterface(
    gradio_chat,
    title="🏥 MediVault Knowledge Assistant",
    description="Ask anything about MediVault's policies, products, or legal documents.",
    examples=[
        "How long is parental leave for primary caregivers?",
        "Can I work fully remotely?",
        "What is the pricing for MediVault Core?",
        "How quickly must MediVault notify me of a breach?",
    ],
)

# demo.launch()     # ← Uncomment to launch in your environment
print("✅ Gradio UI ready — uncomment demo.launch() to start")


### 🏋️ Stage 3 Exercise

The current pipeline rewrites the query even on the first turn (where no rewriting is needed).
Optimise it:

1. Add a check: if `len(history) == 0`, skip the rewrite step and use the original question directly.
2. Measure how many tokens this saves per first-turn query (use `tiktoken`).
3. Add a `verbose=True` flag that prints the rewritten query to the console so you can audit it.

**Bonus**: Implement streaming responses — instead of returning the full answer at once,
use `rag_chain.stream()` and yield token by token to Gradio.


---
# 🟠 Stage 4 — Evaluating RAG: Metrics That Matter

## 🎯 Target
Build a rigorous automated evaluation framework for your RAG pipeline.
By the end of this stage you will:
- Understand the three axes of RAG quality: retrieval, faithfulness, answer quality
- Implement a RAGAS-inspired evaluator using LLM-as-judge
- Generate a test set, run batch evaluation, and interpret results
- Know which metric to optimise for which failure mode

---

## 📖 Theory: Why Evaluation Is Hard (and Critical)

RAG systems fail in three distinct ways:

```
1. Retrieval failure  — the right document was never retrieved
   Symptom: answer contains a hallucination or "I don't know"
   Fix: better embeddings, chunk size, or top-k

2. Faithfulness failure — answer contradicts the retrieved context
   Symptom: answer is wrong even though the document had the right info
   Fix: better prompt, lower temperature, or a stronger LLM

3. Relevance failure — answer doesn't actually address the question
   Symptom: answer is correct but off-topic or incomplete
   Fix: better answer prompt, query rewriting
```

You cannot fix what you cannot measure. Evaluation **must** be automated —
hand-checking 50 Q&A pairs is feasible; hand-checking 5,000 is not.

---

## 📖 Theory: The Three RAG Metrics

| Metric | What it measures | How to compute it |
|--------|-----------------|-------------------|
| **Context Recall** | Did we retrieve the right chunks? | Do keywords from the reference answer appear in retrieved chunks? |
| **Faithfulness** | Does the answer stay within the retrieved context? | LLM judge: "Is every claim in the answer supported by the context?" |
| **Answer Relevance** | Does the answer address the question? | LLM judge: scores 1-5 on relevance |

Together these give you a diagnostic: *where* your pipeline is failing.

---


In [ ]:
# Stage 4 setup

import os
import json
import re
from dataclasses import dataclass, field, asdict
from typing import Optional
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()
JUDGE_MODEL = "gpt-4o-mini"   # cheap judge; use gpt-4o for higher-stakes evals

print("✅ Evaluation setup ready")


### 4.1  Test Set Design

A good evaluation set covers multiple *question categories*:

| Category | Description | Example |
|----------|-------------|---------|
| `direct_fact` | Single document, direct lookup | "What is the encryption algorithm?" |
| `numerical` | Requires extracting a number | "How many weeks of leave?" |
| `comparison` | Spans multiple documents | "How does Core pricing compare to Analytics?" |
| `conditional` | Conditional/contextual fact | "What if I want full remote work?" |
| `negative` | Answer is NOT in the knowledge base | "What is MediVault's stock ticker?" |


In [ ]:
@dataclass
class TestCase:
    question:         str
    reference_answer: str
    category:         str
    keywords:         list[str] = field(default_factory=list)
    # populated during evaluation
    retrieved_texts:  list[str] = field(default_factory=list)
    generated_answer: str       = ""
    context_recall:   float     = 0.0
    faithfulness:     float     = 0.0
    answer_relevance: float     = 0.0
    notes:            str       = ""


TEST_SET: list[TestCase] = [
    # ── Direct facts ───────────────────────────────────────────────────────
    TestCase(
        question="How many weeks of paid leave does a primary caregiver receive?",
        reference_answer="Primary caregivers at MediVault receive 16 weeks of fully paid parental leave.",
        category="direct_fact",
        keywords=["16", "weeks", "primary", "paid"],
    ),
    TestCase(
        question="What is MediVault's AES encryption key size?",
        reference_answer="MediVault uses AES-256 encryption for data at rest.",
        category="direct_fact",
        keywords=["AES-256", "256", "encryption", "at rest"],
    ),
    TestCase(
        question="What is the minimum TLS version MediVault requires?",
        reference_answer="MediVault requires TLS 1.3 as the minimum for data in transit.",
        category="direct_fact",
        keywords=["TLS 1.3", "transit"],
    ),
    TestCase(
        question="What is the annual home-office stipend amount?",
        reference_answer="MediVault provides a $500 annual home-office stipend.",
        category="numerical",
        keywords=["500", "$500", "stipend"],
    ),
    # ── Numerical ──────────────────────────────────────────────────────────
    TestCase(
        question="How much does MediVault Analytics cost per provider per month?",
        reference_answer="MediVault Analytics costs $12 per provider per month, in addition to the Core subscription.",
        category="numerical",
        keywords=["12", "$12", "analytics"],
    ),
    TestCase(
        question="How many years does MediVault retain Analytics data?",
        reference_answer="MediVault Analytics retains data for 7 years.",
        category="numerical",
        keywords=["7", "seven", "years", "retention"],
    ),
    # ── Comparison ─────────────────────────────────────────────────────────
    TestCase(
        question="What is the combined monthly cost per provider for both Core and Analytics?",
        reference_answer="Core costs $18 and Analytics costs $12 per provider per month, for a combined total of $30 per provider per month.",
        category="comparison",
        keywords=["30", "$30", "combined", "18", "12"],
    ),
    TestCase(
        question="How does primary caregiver leave compare to secondary caregiver leave?",
        reference_answer="Primary caregivers receive 16 weeks paid leave while secondary caregivers receive 6 weeks paid leave — a difference of 10 weeks.",
        category="comparison",
        keywords=["16", "6", "primary", "secondary"],
    ),
    # ── Conditional ────────────────────────────────────────────────────────
    TestCase(
        question="What must an employee do to be approved for a fully remote arrangement?",
        reference_answer="Full remote requires VP of People approval, at least 12 months of tenure at MediVault, and manager endorsement.",
        category="conditional",
        keywords=["VP", "12 months", "manager", "endorsement"],
    ),
    TestCase(
        question="How quickly must a breach affecting customer PHI be reported?",
        reference_answer="MediVault will notify the customer within 60 hours of discovering a breach affecting their PHI.",
        category="conditional",
        keywords=["60 hours", "breach", "notification"],
    ),
    # ── Negative (out-of-scope) ────────────────────────────────────────────
    TestCase(
        question="What is MediVault's stock ticker symbol?",
        reference_answer="This information is not available in the provided documents.",
        category="negative",
        keywords=[],   # no keywords — answer should be 'I don't know'
    ),
    TestCase(
        question="Who is the current CEO of MediVault?",
        reference_answer="This information is not available in the provided documents.",
        category="negative",
        keywords=[],
    ),
]

print(f"Test set: {len(TEST_SET)} cases across {len(set(t.category for t in TEST_SET))} categories")
from collections import Counter
print(Counter(t.category for t in TEST_SET))


### 4.2  Metric 1 — Context Recall (Keyword-Based)

Simple but effective: do the keywords from the reference answer appear in retrieved chunks?


In [ ]:
def compute_context_recall(test: TestCase) -> float:
    """
    Keyword-based context recall:
    fraction of reference keywords found in retrieved context.

    Advantages: fast, cheap, no API call needed.
    Limitations: misses paraphrases; use LLM-based recall for high-stakes evals.
    """
    if not test.keywords:
        # Negative case: we expect no retrieval; score by whether
        # the generated answer says 'not available' / 'don't know'
        no_info_phrases = ["not available", "don't have", "not in", "no information", "cannot find"]
        return 1.0 if any(p in test.generated_answer.lower() for p in no_info_phrases) else 0.0

    context_lower = " ".join(test.retrieved_texts).lower()
    found = sum(1 for kw in test.keywords if kw.lower() in context_lower)
    return found / len(test.keywords)


### 4.3  Metric 2 & 3 — Faithfulness & Relevance (LLM Judge)

We prompt GPT-4o-mini to act as a judge. This is the "LLM-as-evaluator" pattern.


In [ ]:
FAITHFULNESS_PROMPT = """You are a strict factual auditor for an AI system.

You will be given:
- QUESTION: the user's question
- CONTEXT: the documents retrieved by the AI system
- ANSWER: the answer generated by the AI

Your task: Evaluate whether every factual claim in the ANSWER is directly supported
by the CONTEXT. Claims that go beyond the context — even if true in the real world —
count as unsupported.

Score from 0.0 to 1.0:
- 1.0 = Every claim is directly supported by the context
- 0.7-0.9 = Most claims supported; minor unsupported additions
- 0.4-0.6 = Some claims supported, others not
- 0.0-0.3 = Many claims contradict or go beyond the context

Also provide a one-sentence explanation.

Respond in JSON only:
{{"faithfulness": <float 0-1>, "explanation": "<string>"}}

QUESTION: {question}

CONTEXT:
{context}

ANSWER:
{answer}
"""

RELEVANCE_PROMPT = """You are evaluating how well an AI assistant answered a user's question.

Score the ANSWER from 1 to 5:
- 5 = Directly and completely answers the question; well-structured
- 4 = Answers the main question; minor omissions or slight tangents
- 3 = Partially answers; important aspects missing
- 2 = Tangentially related; does not clearly answer the question
- 1 = Off-topic or refuses to answer when the information was available

Respond in JSON only:
{{"relevance": <int 1-5>, "explanation": "<string>"}}

QUESTION: {question}
ANSWER: {answer}
"""


def judge_faithfulness(test: TestCase) -> tuple[float, str]:
    context = "\n---\n".join(test.retrieved_texts) or "(no context retrieved)"
    prompt = FAITHFULNESS_PROMPT.format(
        question=test.question,
        context=context,
        answer=test.generated_answer,
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    data = json.loads(resp.choices[0].message.content)
    return float(data.get("faithfulness", 0)), data.get("explanation", "")


def judge_relevance(test: TestCase) -> tuple[float, str]:
    prompt = RELEVANCE_PROMPT.format(
        question=test.question,
        answer=test.generated_answer,
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    data = json.loads(resp.choices[0].message.content)
    score = float(data.get("relevance", 1)) / 5.0   # normalise to 0-1
    return score, data.get("explanation", "")


print("✅ Judge functions ready")


### 4.4  Running the Full Evaluation Loop


In [ ]:
# ── We need a retriever + answer generator from Stage 3 ──────────────────
# For self-containedness, define a minimal one here using OpenAI directly.
# In practice, import your Stage 3 rag_chain and call it here.

from sentence_transformers import SentenceTransformer
import chromadb as chromadb_mod

_embed_model = SentenceTransformer("all-MiniLM-L6-v2")
_chroma = chromadb_mod.PersistentClient(path="/tmp/medivault_chroma")
try:
    _col = _chroma.get_collection("knowledge_base")
    print(f"✅ Retrieved existing collection: {_col.count()} chunks")
except Exception as e:
    print(f"⚠️  Run Stage 2 first to create the ChromaDB collection. ({e})")
    _col = None


ANSWER_SYSTEM = """You are a helpful assistant for MediVault.
Answer the question using ONLY the context below.
If the answer is not in the context, say "This information is not available in the provided documents."
Context:
{context}
"""

def evaluate_test_case(test: TestCase) -> TestCase:
    """Run retrieval + generation + all three metrics for one test case."""

    # 1. Retrieve
    if _col:
        qvec = _embed_model.encode([test.question]).tolist()
        results = _col.query(query_embeddings=qvec, n_results=4,
                             include=["documents"])
        test.retrieved_texts = results["documents"][0]
    else:
        test.retrieved_texts = []

    # 2. Generate
    context = "\n\n".join(test.retrieved_texts) or "(no documents retrieved)"
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM.format(context=context)},
            {"role": "user",   "content": test.question},
        ],
        temperature=0,
    )
    test.generated_answer = resp.choices[0].message.content

    # 3. Evaluate
    test.context_recall   = compute_context_recall(test)
    test.faithfulness, _  = judge_faithfulness(test)
    test.answer_relevance, _ = judge_relevance(test)

    return test


# ── Run evaluation ────────────────────────────────────────────────────────
print("Running evaluation (this calls the LLM for each test case)...")
print()

evaluated = []
for i, test in enumerate(TEST_SET):
    print(f"[{i+1:2d}/{len(TEST_SET)}] {test.category:15s} {test.question[:60]}...")
    result = evaluate_test_case(test)
    evaluated.append(result)

print("\n✅ Evaluation complete")


In [ ]:
# ── Results summary ──────────────────────────────────────────────────────

from collections import defaultdict

def print_results_table(results: list[TestCase]):
    print(f"\n{'='*80}")
    print(f"  RAG EVALUATION REPORT — MediVault Knowledge Assistant")
    print(f"{'='*80}\n")

    # Per-case table
    print(f"{'Q':<3} {'Category':<15} {'Recall':>7} {'Faith.':>7} {'Relev.':>7}  Question")
    print("─" * 80)
    for i, t in enumerate(results):
        avg = (t.context_recall + t.faithfulness + t.answer_relevance) / 3
        flag = "⚠️ " if avg < 0.6 else "  "
        print(f"{i+1:<3} {t.category:<15} {t.context_recall:7.2f} {t.faithfulness:7.2f} "
              f"{t.answer_relevance:7.2f}  {flag}{t.question[:45]}...")

    # Aggregate by category
    print(f"\n{'─'*80}")
    print("  AGGREGATES BY CATEGORY")
    print(f"{'─'*80}")

    by_cat = defaultdict(list)
    for t in results:
        by_cat[t.category].append(t)

    print(f"{'Category':<18} {'N':>3} {'Recall':>8} {'Faithfulness':>13} {'Relevance':>10} {'Overall':>8}")
    print("─" * 65)
    for cat, cases in sorted(by_cat.items()):
        n   = len(cases)
        rec = sum(c.context_recall   for c in cases) / n
        fai = sum(c.faithfulness     for c in cases) / n
        rel = sum(c.answer_relevance for c in cases) / n
        ove = (rec + fai + rel) / 3
        print(f"{cat:<18} {n:>3} {rec:>8.2f} {fai:>13.2f} {rel:>10.2f} {ove:>8.2f}")

    # Overall
    n = len(results)
    overall_rec = sum(t.context_recall   for t in results) / n
    overall_fai = sum(t.faithfulness     for t in results) / n
    overall_rel = sum(t.answer_relevance for t in results) / n
    overall     = (overall_rec + overall_fai + overall_rel) / 3

    print("─" * 65)
    print(f"{'OVERALL':<18} {n:>3} {overall_rec:>8.2f} {overall_fai:>13.2f} "
          f"{overall_rel:>10.2f} {overall:>8.2f}")
    print()

    # Worst-performing cases
    sorted_by_overall = sorted(
        results,
        key=lambda t: (t.context_recall + t.faithfulness + t.answer_relevance) / 3
    )
    print("  🔴 WORST-PERFORMING CASES (bottom 3)")
    for t in sorted_by_overall[:3]:
        avg = (t.context_recall + t.faithfulness + t.answer_relevance) / 3
        print(f"   [{avg:.2f}] {t.question}")
        print(f"          Generated: {t.generated_answer[:100]}...")
        print()


print_results_table(evaluated)


### 4.5  Interpreting Results: A Diagnostic Framework

```
Low Context Recall
└─► Fix retrieval: better embeddings, larger k, smaller chunks,
    hybrid (sparse + dense) search, or add BM25 fallback.

Low Faithfulness (but high recall)
└─► Fix generation: stronger prompt ("only use context"),
    lower temperature, or a more instruction-following model.

Low Relevance (but high recall + faithfulness)
└─► Fix answer quality: add chain-of-thought to prompt,
    ask for structured output, improve system prompt.

All metrics low on 'negative' questions
└─► Add a confidence threshold: if max similarity < 0.35, return
    "I don't have this information" without calling the LLM.
```

---

### 🏋️ Stage 4 Exercise

1. Add a `"multi_hop"` category test case that requires combining information
   from *two* different documents (e.g., "What is the combined monthly cost
   of both products, and how does the DPA affect their usage?").
2. Run the evaluator on it. Is faithfulness or recall the bottleneck?
3. Modify the retrieval to use `top_k=6` for multi-hop questions and re-evaluate.


---
# 🔴 Stage 5 — Advanced RAG: Reranking, Hybrid Search & Production Hardening

## 🎯 Target
Apply production-grade techniques that close the gap between demo and deployment.
By the end of this stage you will:
- Implement **cross-encoder reranking** to improve retrieval precision
- Build **hybrid search** (dense vectors + BM25 keyword matching)
- Add **guardrails**: confidence thresholding and hallucination detection
- Understand the performance vs cost trade-offs at each step

---

## 📖 Theory: The Two-Stage Retrieval Pattern

Standard RAG retrieves top-k chunks with an embedding model (bi-encoder).
This is fast but imprecise — the bi-encoder encodes query and document *separately*,
so subtle relevance nuances are lost.

**Two-stage retrieval:**

```
Stage A — Recall (cheap bi-encoder)
  Query → embed → ANN search → top-20 candidates (fast, ~10ms)

Stage B — Precision (expensive cross-encoder)
  (Query, candidate) pairs → cross-encoder score → reranked top-5
  (slow, but only 20 pairs × ~5ms = 100ms total)
```

**Cross-encoders** process the query and document *together*, so they capture
exact term overlap, negations, and coreference. They're too slow for a full corpus
but perfect for reranking a short candidate list.

---

## 📖 Theory: Hybrid Search (Dense + Sparse)

| Search type | Strength | Weakness |
|------------|---------|---------|
| Dense (embeddings) | Semantic similarity | Misses exact terms (product codes, names) |
| Sparse (BM25/TF-IDF) | Exact term match | Misses paraphrases |
| **Hybrid (both)** | **Best of both** | **Slightly more complex** |

A common formula: `final_score = α × dense_score + (1-α) × sparse_score`

When `α = 0.7`, dense search dominates but rare terms still get boosted by BM25.

---


In [ ]:
# Stage 5 setup

import os
import math
import json
import heapq
import numpy as np
from typing import NamedTuple
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv
import chromadb

load_dotenv(override=True)
client = OpenAI()

EMBED_MODEL   = "all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # top reranker for RAG

embed_model   = SentenceTransformer(EMBED_MODEL)
cross_encoder = CrossEncoder(CROSS_ENCODER)

print(f"✅ Bi-encoder  : {EMBED_MODEL}")
print(f"✅ Cross-encoder: {CROSS_ENCODER}")


In [ ]:
# Load our knowledge base chunks (from Stage 2 ChromaDB)

chroma = chromadb.PersistentClient(path="/tmp/medivault_chroma")
try:
    col = chroma.get_collection("knowledge_base")
    raw = col.get(include=["documents", "metadatas", "embeddings"])
    CORPUS_TEXTS    = raw["documents"]
    CORPUS_METAS    = raw["metadatas"]
    CORPUS_VECTORS  = np.array(raw["embeddings"])
    print(f"✅ Corpus loaded: {len(CORPUS_TEXTS)} chunks")
except Exception as e:
    print(f"⚠️  Run Stage 2 first. ({e})")
    CORPUS_TEXTS, CORPUS_METAS, CORPUS_VECTORS = [], [], np.array([])


### 5.1  BM25 From Scratch

BM25 (Best Match 25) is the industry-standard keyword search algorithm.
Let's implement it so the math is transparent.


In [ ]:
import re
from collections import Counter

class BM25:
    """
    BM25 sparse retrieval.
    Parameters k1=1.5 and b=0.75 are standard defaults from the 1994 paper.
    """
    def __init__(self, corpus: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self.N  = len(corpus)
        self.tokenised = [self._tokenise(doc) for doc in corpus]
        self.doc_lengths = [len(t) for t in self.tokenised]
        self.avg_dl = sum(self.doc_lengths) / self.N

        # Build inverted index: term → {doc_id: term_frequency}
        self.idf: dict[str, float] = {}
        self.tf:  list[dict[str, int]] = []
        all_terms = set()
        for tokens in self.tokenised:
            tf_doc = Counter(tokens)
            self.tf.append(tf_doc)
            all_terms.update(tf_doc.keys())

        # IDF: log((N - df + 0.5) / (df + 0.5) + 1)
        for term in all_terms:
            df = sum(1 for tf_doc in self.tf if term in tf_doc)
            self.idf[term] = math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    @staticmethod
    def _tokenise(text: str) -> list[str]:
        return re.findall(r'\b\w+\b', text.lower())

    def score(self, query: str, doc_id: int) -> float:
        tokens = self._tokenise(query)
        dl     = self.doc_lengths[doc_id]
        score  = 0.0
        for term in tokens:
            if term not in self.idf:
                continue
            tf  = self.tf[doc_id].get(term, 0)
            idf = self.idf[term]
            numerator   = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avg_dl)
            score += idf * numerator / denominator
        return score

    def retrieve(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        """Returns [(doc_id, score)] sorted by score descending."""
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]


# Build BM25 index
if CORPUS_TEXTS:
    bm25 = BM25(CORPUS_TEXTS)
    print(f"✅ BM25 index built on {len(CORPUS_TEXTS)} documents")
    print(f"   Vocabulary size: {len(bm25.idf)} terms")

    # Quick sanity check
    query = "AES encryption at rest"
    results = bm25.retrieve(query, top_k=3)
    print(f"\nBM25 top-3 for '{query}':")
    for doc_id, score in results:
        print(f"  [{score:.3f}] {CORPUS_TEXTS[doc_id][:80]}...")


### 5.2  Hybrid Search (Dense + BM25)


In [ ]:
def dense_retrieve(query: str, top_k: int = 20) -> list[tuple[int, float]]:
    """Bi-encoder retrieval; returns (doc_id, cosine_similarity) pairs."""
    qvec = embed_model.encode([query])[0]
    # Cosine similarity with all corpus vectors
    norms = np.linalg.norm(CORPUS_VECTORS, axis=1) * np.linalg.norm(qvec)
    sims  = np.dot(CORPUS_VECTORS, qvec) / np.maximum(norms, 1e-10)
    top   = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i])) for i in top]


def hybrid_retrieve(
    query:    str,
    top_k:    int   = 20,
    alpha:    float = 0.7,   # weight for dense; (1-alpha) for BM25
) -> list[tuple[int, float]]:
    """
    Hybrid retrieval:
    1. Get dense scores for all docs (normalised 0-1)
    2. Get BM25 scores for all docs (normalised 0-1)
    3. Combine: final = alpha * dense + (1-alpha) * bm25
    4. Return top_k by combined score
    """
    # Dense scores (already cosine similarities in [-1, 1] → clip to [0, 1])
    qvec = embed_model.encode([query])[0]
    norms = np.linalg.norm(CORPUS_VECTORS, axis=1) * np.linalg.norm(qvec)
    dense_scores = np.dot(CORPUS_VECTORS, qvec) / np.maximum(norms, 1e-10)
    dense_scores = np.clip(dense_scores, 0, 1)

    # BM25 scores (normalise by max)
    bm25_raw = np.array([bm25.score(query, i) for i in range(len(CORPUS_TEXTS))])
    bm25_max = bm25_raw.max() or 1.0
    bm25_scores = bm25_raw / bm25_max

    # Combine
    combined = alpha * dense_scores + (1 - alpha) * bm25_scores
    top = np.argsort(combined)[::-1][:top_k]
    return [(int(i), float(combined[i])) for i in top]


# ── Compare dense vs hybrid retrieval ────────────────────────────────────

if CORPUS_TEXTS:
    test_query = "TLS version MediVault"

    print(f"Query: '{test_query}'\n")
    print("── Dense retrieval top-4 ──")
    for doc_id, score in dense_retrieve(test_query, top_k=4):
        print(f"  [{score:.3f}] {CORPUS_TEXTS[doc_id][:80]}...")

    print("\n── Hybrid retrieval top-4 (α=0.7) ──")
    for doc_id, score in hybrid_retrieve(test_query, top_k=4):
        print(f"  [{score:.3f}] {CORPUS_TEXTS[doc_id][:80]}...")


### 5.3  Cross-Encoder Reranking

We retrieve 20 candidates with hybrid search, then rerank them using the cross-encoder.


In [ ]:
def retrieve_and_rerank(
    query:          str,
    initial_k:      int   = 15,   # candidates from hybrid search
    final_k:        int   = 4,    # after reranking
    alpha:          float = 0.7,
) -> list[dict]:
    """
    Full two-stage retrieval:
    Stage A — Hybrid search (fast, high recall)
    Stage B — Cross-encoder rerank (slow, high precision)
    """
    # Stage A: recall
    candidates = hybrid_retrieve(query, top_k=initial_k, alpha=alpha)

    # Stage B: rerank
    pairs = [(query, CORPUS_TEXTS[doc_id]) for doc_id, _ in candidates]
    ce_scores = cross_encoder.predict(pairs).tolist()

    reranked = sorted(
        [(candidates[i][0], ce_scores[i]) for i in range(len(candidates))],
        key=lambda x: x[1],
        reverse=True,
    )

    return [
        {
            "doc_id":   doc_id,
            "ce_score": round(score, 4),
            "text":     CORPUS_TEXTS[doc_id],
            "source":   CORPUS_METAS[doc_id]["source"],
        }
        for doc_id, score in reranked[:final_k]
    ]


# ── Test reranking ─────────────────────────────────────────────────────────

if CORPUS_TEXTS:
    tricky_query = "How quickly must MediVault inform customers about a security incident?"

    print(f"Query: '{tricky_query}'\n")
    print("── Reranked results ──")
    results = retrieve_and_rerank(tricky_query, initial_k=10, final_k=4)
    for r in results:
        print(f"  CE score: {r['ce_score']:6.3f}  Source: {r['source']}")
        print(f"  {r['text'][:120]}...")
        print()


### 5.4  Confidence Thresholding (Guardrail)

If no retrieved document is sufficiently relevant, the LLM should abstain
rather than hallucinate. We implement this with a minimum CE score threshold.


In [ ]:
CONFIDENCE_THRESHOLD = 0.0   # Cross-encoder scores are log-odds; 0.0 ~ 50% confidence
NO_CONTEXT_RESPONSE  = (
    "I don't have enough information in the MediVault knowledge base to answer "
    "this question reliably. Please consult the relevant team directly."
)

ANSWER_SYSTEM = """You are a helpful, precise internal assistant for MediVault.
Answer the user's question using ONLY the retrieved context below.
- Always cite the source document.
- If the context does not contain the answer, say so — never guess.
- Be concise (2-4 sentences).

Context:
{context}
"""

def production_rag(query: str, verbose: bool = False) -> str:
    """
    Production-grade RAG:
    1. Hybrid retrieve → cross-encoder rerank
    2. Confidence check → abstain if below threshold
    3. Generate with context-grounded prompt
    """
    if not CORPUS_TEXTS:
        return "Knowledge base not loaded. Run Stage 2 first."

    # Step 1: Retrieve + rerank
    results = retrieve_and_rerank(query, initial_k=12, final_k=4)

    if verbose:
        print(f"\n[DEBUG] Top CE scores: {[r['ce_score'] for r in results]}")

    # Step 2: Confidence gate
    best_score = results[0]["ce_score"] if results else -99
    if best_score < CONFIDENCE_THRESHOLD:
        if verbose:
            print(f"[DEBUG] Below threshold ({best_score:.3f} < {CONFIDENCE_THRESHOLD}) → abstaining")
        return NO_CONTEXT_RESPONSE

    # Step 3: Generate
    context = "\n\n---\n\n".join(
        f"[{r['source']}]\n{r['text']}" for r in results
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM.format(context=context)},
            {"role": "user",   "content": query},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content


# ── Test the production pipeline ──────────────────────────────────────────

test_queries = [
    ("In-scope",  "What is the breach notification timeline?"),
    ("In-scope",  "How long can an employee take parental leave?"),
    ("Out-of-scope", "What is MediVault's annual revenue?"),
    ("Out-of-scope", "Who founded MediVault?"),
]

if CORPUS_TEXTS:
    for label, query in test_queries:
        print(f"[{label}] {query}")
        answer = production_rag(query, verbose=True)
        print(f"Answer: {answer[:200]}")
        print()


### 5.5  Hallucination Detection (Post-Generation Guard)

Even with a good retriever, the LLM may occasionally confabulate.
We add a lightweight post-generation check using the same LLM.


In [ ]:
HALLUCINATION_CHECK_PROMPT = """
You are a factual auditor. Your task is to check whether an AI-generated answer
is fully supported by the provided source context.

Rate the answer:
- "SUPPORTED"    — every factual claim in the answer comes from the context
- "UNSUPPORTED"  — one or more claims go beyond or contradict the context
- "PARTIAL"      — some claims are supported, others are not

Respond in JSON:
{{"verdict": "SUPPORTED|UNSUPPORTED|PARTIAL", "issue": "<brief description or null>"}}

CONTEXT:
{context}

ANSWER:
{answer}
"""

def check_hallucination(answer: str, context_chunks: list[dict]) -> dict:
    """Post-generation hallucination check. Returns verdict dict."""
    context = "\n---\n".join(c["text"] for c in context_chunks)
    prompt  = HALLUCINATION_CHECK_PROMPT.format(context=context, answer=answer)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


def production_rag_with_guard(query: str) -> dict:
    """Full pipeline with hallucination guard. Returns structured result."""
    if not CORPUS_TEXTS:
        return {"answer": "Knowledge base not loaded.", "verdict": "N/A"}

    chunks = retrieve_and_rerank(query, initial_k=12, final_k=4)
    best_score = chunks[0]["ce_score"] if chunks else -99

    if best_score < CONFIDENCE_THRESHOLD:
        return {"answer": NO_CONTEXT_RESPONSE, "verdict": "ABSTAINED", "ce_score": best_score}

    context = "\n\n---\n\n".join(f"[{c['source']}]\n{c['text']}" for c in chunks)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM.format(context=context)},
            {"role": "user",   "content": query},
        ],
        temperature=0,
    )
    answer = resp.choices[0].message.content

    # Post-generation check
    verdict_data = check_hallucination(answer, chunks)

    return {
        "question": query,
        "answer":   answer,
        "verdict":  verdict_data["verdict"],
        "issue":    verdict_data.get("issue"),
        "ce_score": best_score,
        "sources":  [c["source"] for c in chunks],
    }


if CORPUS_TEXTS:
    result = production_rag_with_guard("What data formats can Analytics export to?")
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"   Verdict: {result['verdict']}  |  CE score: {result['ce_score']:.3f}")
    if result.get("issue"):
        print(f"   Issue: {result['issue']}")


### 5.6  Architecture Summary: What We Built

```
┌─────────────────────────────────────────────────────────────────────┐
│                  PRODUCTION RAG PIPELINE                            │
│                                                                     │
│  User question                                                      │
│       │                                                             │
│       ▼  (if multi-turn)                                            │
│  ┌─────────────┐   LLM     ┌──────────────────────┐                │
│  │ Query       │ ────────► │ Standalone query      │                │
│  │ Rewriter    │           └──────────┬───────────┘                │
│  └─────────────┘                      │                             │
│                                       ▼                             │
│  ┌────────────────────────────────────────────────────────┐        │
│  │ RETRIEVAL LAYER                                        │        │
│  │   ┌─────────────┐     ┌─────────────┐                 │        │
│  │   │ Dense search│     │ BM25 search │                 │        │
│  │   │ (bi-encoder)│     │ (keyword)   │                 │        │
│  │   └──────┬──────┘     └──────┬──────┘                 │        │
│  │          └──────────┬────────┘                         │        │
│  │                     ▼  α·dense + (1-α)·BM25            │        │
│  │            Top-15 candidates                           │        │
│  │                     │                                  │        │
│  │                     ▼  Cross-encoder                   │        │
│  │            Reranked top-4                              │        │
│  └────────────────────────────────────────────────────────┘        │
│                       │                                             │
│                       ▼  Confidence gate                            │
│              Score < threshold? ──► ABSTAIN                        │
│                       │                                             │
│                       ▼                                             │
│  ┌─────────────────────────────────┐                               │
│  │ GENERATION                      │                               │
│  │ System prompt + context + query │                               │
│  │ → LLM → Answer                  │                               │
│  └──────────────┬──────────────────┘                               │
│                 │                                                   │
│                 ▼  Hallucination guard                              │
│  Verdict: SUPPORTED / PARTIAL / UNSUPPORTED                        │
│                 │                                                   │
│                 ▼                                                   │
│           Final answer (+ sources + verdict)                       │
└─────────────────────────────────────────────────────────────────────┘
```

---

### 🏋️ Stage 5 Exercise

**A. Tune the hybrid alpha**
Run the Stage 4 evaluator on your production pipeline with `alpha = 0.3, 0.5, 0.7, 0.9`.
Plot context recall vs alpha. At what alpha does your test set score best?

**B. Dynamic threshold**
Instead of a fixed `CONFIDENCE_THRESHOLD`, compute a per-query threshold as
`mean(ce_scores) - 0.5 * std(ce_scores)`. If the top result is below this,
abstain. Does this reduce false abstentions?

**C. Streaming UI**
Integrate `production_rag_with_guard` into a Gradio `ChatInterface`
that also displays the sources and verdict in a collapsible section below the answer.


---
# 📚 Appendix: RAG Decision Guide

## When to use which technique

| Situation | Recommended approach |
|-----------|---------------------|
| < 50 documents, prototype | Stage 1: in-memory cosine sim |
| 50-10k documents, stable corpus | Stage 2: ChromaDB + HF embeddings |
| Multi-turn chatbot | Stage 3: history-aware rewriting |
| Going to production | Stage 4: evaluation first, then optimise |
| High-precision requirements | Stage 5: hybrid + reranking + guards |
| Very large corpus (>1M chunks) | Upgrade to Pinecone/Weaviate/pgvector |

## Common RAG Failure Modes & Fixes

| Symptom | Root cause | Fix |
|---------|-----------|-----|
| "I don't have this info" when doc exists | Embedding mismatch | Try different embedding model; reduce chunk size |
| Correct context retrieved but wrong answer | Faithfulness failure | Stronger system prompt; add "only use context" |
| Correct answer but ignores follow-up | No history-aware rewriting | Add Stage 3 query rewriting |
| Hallucination slips through | No post-generation guard | Add Stage 5 hallucination checker |
| Out-of-scope answers confidently | No confidence threshold | Add CE score gate |
| Slow responses | Reranking on too many candidates | Reduce `initial_k`; cache BM25 index |

## Recommended Reading
- **RAG Paper**: Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (2020)
- **RAGAS**: Shahul et al., "RAGAS: Automated Evaluation of Retrieval Augmented Generation" (2023)
- **Lost in the Middle**: Liu et al., "Lost in the Middle: How Language Models Use Long Contexts" (2023)
- **HyDE**: Gao et al., "Precise Zero-Shot Dense Retrieval without Relevance Labels" (2022) — hypothetical document embeddings, another query rewriting technique
